# Full lipsync demo on Colab GPU

**Only manual steps in Colab:** **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

Push your latest code to **GitHub `main`** first. If the repo is already under `/content` from a previous session, the first cell runs **`git pull --ff-only`** so you test the same code you just pushed.

When the last cell prints **OPEN IN BROWSER:** `https://…trycloudflare.com` — open that link in a **new tab**, allow the **microphone**, upload a photo → **Set as Avatar** → **Start** → speak. Hard-refresh (`Ctrl+F5`) if the page looks cached.

The notebook clones your repo if needed, installs dependencies, starts FastAPI on GPU, and opens a public HTTPS tunnel (no signup).


In [1]:
# Auto-setup: git pull existing clone, or shallow-clone to /content/avatar
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/muhammademraan33-svg/Real-Time-Animated-Avatar-Creation.git"


def _is_full_repo(p: Path) -> bool:
    return (
        (p / "server" / "main.py").is_file()
        and (p / "static" / "index.html").is_file()
    )


def _git_pull_if_possible(repo: Path) -> None:
    if not (repo / ".git").is_dir():
        return
    subprocess.run(["git", "-C", str(repo), "remote", "set-url", "origin", REPO_URL], check=False)
    r = subprocess.run(
        ["git", "-C", str(repo), "pull", "--ff-only"],
        capture_output=True,
        text=True,
    )
    if r.returncode == 0:
        print("git pull: OK (latest origin)")
    else:
        print("git pull skipped or failed. stderr:", (r.stderr or "")[-300:])


def get_project() -> Path:
    preferred = [
        Path("/content/avatar"),
        Path("/content/Real-Time-Animated-Avatar-Creation"),
    ]
    for p in preferred:
        if p.is_dir():
            _git_pull_if_possible(p)
        if _is_full_repo(p):
            print("Using repo:", p)
            return p.resolve()
    for p in Path("/content").iterdir():
        if not p.is_dir():
            continue
        _git_pull_if_possible(p)
        if _is_full_repo(p):
            print("Using repo:", p)
            return p.resolve()
    dest = Path("/content/avatar")
    if dest.exists():
        shutil.rmtree(dest)
    subprocess.run(
        ["git", "clone", "--depth", "1", REPO_URL, str(dest)],
        check=True,
    )
    print("Cloned to", dest)
    return dest.resolve()


PROJECT = get_project()
print("PROJECT =", PROJECT)


Cloned to /content/avatar
PROJECT = /content/avatar


In [2]:
import subprocess

subprocess.run(["apt-get", "update", "-qq"], check=False)
subprocess.run(
    ["apt-get", "install", "-qq", "-y", "git", "ffmpeg", "libsndfile1", "libgl1-mesa-glx"],
    check=False,
)
print("apt deps OK")


apt deps OK


In [3]:
import subprocess
import sys

if "get_project" not in globals():
    raise RuntimeError("Run all cells from the top (Runtime → Run all), or run the setup cell first.")

PROJECT = get_project()

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "numpy>=1.24,<2.0", "huggingface_hub", "tqdm", "requests",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements.txt"),
])
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


git pull: OK (latest origin)
Using repo: /content/avatar
torch 2.10.0+cu128 | CUDA: True Tesla T4


In [4]:
import os
from pathlib import Path

if "get_project" not in globals():
    raise RuntimeError("Run all cells from the top.")

PROJECT = get_project()

from huggingface_hub import hf_hub_download

ckpt_dir = PROJECT / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = hf_hub_download(
    repo_id="Nekochu/Wav2Lip",
    filename="wav2lip_gan.pth",
    local_dir=str(ckpt_dir),
)
os.environ["CHECKPOINT_PATH"] = str(Path(checkpoint_path).resolve())
print("CHECKPOINT_PATH =", os.environ["CHECKPOINT_PATH"])


git pull: OK (latest origin)
Using repo: /content/avatar


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


wav2lip_gan.pth:   0%|          | 0.00/436M [00:00<?, ?B/s]

CHECKPOINT_PATH = /content/avatar/checkpoints/wav2lip_gan.pth


In [5]:
import os
import subprocess
import time

import requests

if "get_project" not in globals():
    raise RuntimeError("Run all cells from the top.")

PROJECT = get_project()

subprocess.run(["pkill", "-f", "uvicorn"], check=False)
subprocess.run(["pkill", "-f", "cloudflared"], check=False)
time.sleep(1)

log_path = "/tmp/uvicorn_avatar.log"
log = open(log_path, "w")
env = {**os.environ, "CHECKPOINT_PATH": os.environ.get("CHECKPOINT_PATH", "")}
proc = subprocess.Popen(
    [os.sys.executable, "-m", "uvicorn", "server.main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=str(PROJECT),
    stdout=log,
    stderr=subprocess.STDOUT,
    env=env,
    start_new_session=True,
)
print("uvicorn PID:", proc.pid, "| log:", log_path)

for i in range(180):
    time.sleep(2)
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=3)
        if r.status_code == 200:
            print("Server ready:", r.json())
            break
    except Exception as e:
        if i > 0 and i % 15 == 0:
            print("Still waiting for /health … (GPU model load can take a few minutes)", e)
else:
    log.seek(0)
    print(log.read()[-4000:])
    raise RuntimeError("Server did not become healthy in time — see log above")


git pull: OK (latest origin)
Using repo: /content/avatar
uvicorn PID: 1819 | log: /tmp/uvicorn_avatar.log
Server ready: {'status': 'ok', 'model_loaded': True, 'has_avatar': False, 'total_frames_generated': 0, 'last_inference_ms': 0.0, 'device': 'cuda', 'gpu': {'name': 'Tesla T4', 'memory_allocated_mb': 77.0, 'memory_reserved_mb': 117.4}}


In [6]:
import os
import re
import subprocess
import threading
import time
import urllib.request

if "get_project" not in globals():
    raise RuntimeError("Run all cells from the top.")

BIN = "/content/cloudflared"
if not os.path.isfile(BIN):
    url = "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    urllib.request.urlretrieve(url, BIN)
    os.chmod(BIN, 0o755)

print("Starting tunnel (watch for https://…trycloudflare.com)…\n")

p = subprocess.Popen(
    [BIN, "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public = {"url": None}
URL_RE = re.compile(r"https://[a-zA-Z0-9.-]+[.]trycloudflare[.]com")


def reader():
    for line in p.stdout:
        print(line, end="")
        m = URL_RE.search(line)
        if m:
            public["url"] = m.group(0)


threading.Thread(target=reader, daemon=True).start()

for _ in range(90):
    time.sleep(1)
    if public["url"]:
        break

if public["url"]:
    u = public["url"]
    print("\n" + "=" * 60)
    print("OPEN IN BROWSER:", u)
    print("=" * 60)
else:
    print("Could not parse tunnel URL — read Cloudflare output above for https://…trycloudflare.com")


Starting tunnel (watch for https://…trycloudflare.com)…

2026-04-14T18:46:25Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-14T18:46:25Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-04-14T18:46:29Z INF +--------------------------------------------------------------------------------------------+
2026-04-14T18:46:29Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-04-14T18:46:29Z INF